# 🚨 DBSCAN: Detectando Anomalías en Telemetría SCADA
## Módulo 4 · Día 2 · Machine Learning No Supervisado · Capacitación SLB

**Instructor: David Ponce**

---

### 🎯 ¿Qué aprenderemos hoy?

Ayer usamos **K-Means** para agrupar pozos. Hoy enfrentamos un problema más real:
los sensores SCADA **fallan**. Transmisores que se desconectan, cables que captan
interferencia, medidores que se atascan. Esos datos corruptos **contaminan**
cualquier modelo que construyamos.

**DBSCAN** no solo agrupa — también **detecta qué datos son BASURA** y los aísla
con la etiqueta `-1`.

### 📋 Objetivos de la sesión:
1. Entender por qué K-Means falla cuando hay sensores dañados
2. Aprender qué es la **densidad** y cómo DBSCAN la usa para encontrar clusters
3. Calibrar el parámetro **Epsilon (ε)** usando el gráfico de k-distancias
4. Entrenar DBSCAN y detectar datos corruptos como **ruido (-1)**
5. Visualizar e interpretar los resultados

> 💡 **Tip:** Cada celda de código está comentada línea por línea. Lee los comentarios
> mientras ejecutas. Si algo no te queda claro, ¡pregunta!

---
## 🧩 PARTE 1: Importando las herramientas que necesitamos

Antes de tocar datos, cargamos las librerías. Piensa en esto como sacar
las herramientas del maletín antes de empezar a trabajar en el pozo.

### 📦 Celda 1: Importación de librerías

Cada `import` trae una herramienta específica. Vamos a entender **qué hace
cada una** antes de usarla.

In [ ]:
# ============================================
# CELDA 1: Importación de librerías
# ============================================

# ─── Herramientas de cálculo y datos ───
import numpy as np
#   ↑ 'numpy' = Numerical Python.
#   ↑ 'as np' es un alias: escribimos 'np' en vez de 'numpy' para ahorrar tecleo.
#   ↑ Lo usaremos para np.sort() al calibrar epsilon.

import pandas as pd
#   ↑ 'pandas' = Panel Data. Es LA librería para trabajar con tablas (DataFrames).
#   ↑ 'as pd' = alias. Lo usamos para pd.read_csv() y manejar el dataset.

# ─── Herramientas de visualización ───
import matplotlib.pyplot as plt
#   ↑ 'matplotlib' es la librería de gráficos más usada en Python.
#   ↑ '.pyplot' es el módulo que nos da funciones tipo plt.plot(), plt.show().
#   ↑ 'as plt' = alias estándar.

import seaborn as sns
#   ↑ 'seaborn' hace gráficos estadísticos más bonitos que matplotlib puro.
#   ↑ Lo usamos para sns.scatterplot() con colores por cluster.
#   ↑ 'as sns' = alias estándar (viene de 'seaborn' pronunciado como 's' + 'n' + 's').

# ─── Herramientas de Machine Learning ───
from sklearn.preprocessing import StandardScaler
#   ↑ 'sklearn' = Scikit-Learn, la librería de ML más popular de Python.
#   ↑ '.preprocessing' = módulo para transformar datos ANTES de modelar.
#   ↑ 'StandardScaler' = EL escalador Z-Score. Transforma cada variable
#   ↑   para que tenga media=0 y desviación estándar=1.
#   ↑   ¡IGUAL DE CRÍTICO que en K-Means! Sin esto, DBSCAN no funciona.

from sklearn.cluster import DBSCAN
#   ↑ '.cluster' = módulo de algoritmos de agrupamiento.
#   ↑ 'DBSCAN' = Density-Based Spatial Clustering of Applications with Noise.
#   ↑   El protagonista del día. Agrupa por densidad y detecta ruido.

from sklearn.neighbors import NearestNeighbors
#   ↑ '.neighbors' = módulo para calcular vecinos más cercanos.
#   ↑ 'NearestNeighbors' = calcula la distancia de cada punto a sus k vecinos.
#   ↑   Lo usamos EXCLUSIVAMENTE para calibrar epsilon (gráfico k-distance).
#   ↑   Esta librería NO la usamos en K-Means. Es nueva hoy.

# ─── Configuración estética ───
sns.set_theme(style="whitegrid")
#   ↑ Aplica un tema visual a TODOS los gráficos de seaborn y matplotlib.
#   ↑ 'whitegrid' = fondo blanco con líneas de cuadrícula grises.
#   ↑ Esto hace que los gráficos sean más fáciles de leer.

print("✅ Todas las librerías importadas correctamente.")


### 🔍 ¿Qué hace cada línea? — Explicación detallada

| Librería | ¿Qué es? | ¿Para qué la usamos HOY? |
|----------|----------|---------------------------|
| `numpy` | Cálculo numérico profesional | `np.sort()` para ordenar las distancias en el gráfico k-distance |
| `pandas` | Manejo de tablas (DataFrames) | Cargar el CSV con `pd.read_csv()`, crear columna 'Cluster' |
| `matplotlib.pyplot` | Gráficos base | `plt.plot()`, `plt.axhline()`, `plt.show()` |
| `seaborn` | Gráficos estadísticos bonitos | `sns.scatterplot()` con `hue='Cluster'` para pintar cada grupo de color |
| `StandardScaler` | **El escalador Z-Score** | Transforma los datos a media=0, std=1. ¡OBLIGATORIO! |
| `DBSCAN` | **El algoritmo de densidad** | `fit_predict()` entrena y asigna clusters (0, 1, 2...) y ruido (-1) |
| `NearestNeighbors` | Calculadora de vecinos | Solo para calibrar ε (epsilon) con el gráfico k-distance |

> ⚠️ **¿Notaste?** `NearestNeighbors` es NUEVA. No la usamos en K-Means. Es una
> herramienta auxiliar para calibrar DBSCAN, no un algoritmo de clustering.


---
## 📥 PARTE 2: Cargando los datos de telemetría

El dataset de hoy es `telemetria_anomala.csv`. A diferencia de ayer (que
teníamos 50 pozos con 7 variables), hoy trabajamos con **1 solo pozo**
monitoreado a lo largo del tiempo. El objetivo es detectar lecturas
anómalas dentro de su historial.

El dataset contiene **1,000 registros** con solo 3 columnas:
- `Registro_ID`: número de registro (1 a 1000)
- `WHP_psi`: Presión en Cabeza del Pozo (psi)
- `Qo_bpd`: Caudal de Crudo (barriles por día)

> 🧠 **¿Por qué solo 2 variables?** El dataset real de telemetría de este pozo
> solo mide presión en cabeza y caudal. Pero DBSCAN funciona con **N variables**
> — igual que K-Means. Si tuvieras Water-Cut, GOR, BHP, etc., DBSCAN las
> usaría TODAS.

### 📦 Celda 2: Descargar y cargar el dataset

In [ ]:
# ============================================
# CELDA 2: Descargar y cargar el dataset
# ============================================

# ─── Paso 1: Descargar el archivo desde GitHub ───
!wget -q https://raw.githubusercontent.com/DavidPonce84/machine-learning-course/main/modulo_4_no_supervisado/data/telemetria_anomala.csv -O telemetria_anomala.csv
#   ↑ '!' le dice a Google Colab: "ejecuta esto como comando del sistema".
#   ↑ Colab corre sobre Linux, así que acepta comandos Linux.
#   ↑ 'wget' = descargador de archivos de internet.
#   ↑ '-q' = "quiet" (silencioso: no muestra barras de progreso).
#   ↑ La URL larga = ubicación del archivo en GitHub.
#   ↑ '-O telemetria_anomala.csv' = "Output": guarda el archivo con este nombre.

# ─── Paso 2: Cargar el CSV en un DataFrame ───
df_scada = pd.read_csv('telemetria_anomala.csv')
#   ↑ 'pd' = pandas (lo importamos arriba).
#   ↑ '.read_csv()' = función que lee archivos CSV y los convierte en DataFrame.
#   ↑ 'df_scada' = nombre que le damos a nuestra tabla. "df" de DataFrame,
#   ↑   "scada" porque son datos de telemetría SCADA.
#   ↑ El DataFrame es como una hoja de Excel dentro de Python.

# ─── Paso 3: Ver las primeras filas ───
df_scada.head()
#   ↑ '.head()' = muestra las primeras 5 filas del DataFrame.
#   ↑ Es como un "vistazo rápido" para verificar que los datos
#   ↑   se cargaron correctamente y las columnas tienen sentido.
#   ↑ Puedes poner .head(10) para ver 10 filas en vez de 5.


### 🔍 Explicación

**`!wget`** — El signo de exclamación `!` es un "escape" que le dice a Colab:
"lo que sigue no es Python, es un comando de Linux". `wget` es una herramienta
clásica de Linux para descargar archivos. El `-O` (Output) define el nombre
con que se guarda localmente.

**`pd.read_csv()`** — Es el "abrelatas" de archivos CSV. Lee el archivo,
detecta automáticamente las comas como separadores, y construye una tabla
(DataFrame) con filas y columnas.

**`.head()`** — Muestra las primeras 5 filas. Es tu primer chequeo: ¿se ven
bien los nombres de columna? ¿Los números tienen sentido? ¿Hay textos donde
debería haber números?

> 🧪 **Prueba:** Cambia `.head()` por `.head(10)` y ejecuta de nuevo. ¿Qué pasa?

---
## 👁️ PARTE 3: Primer vistazo a los datos

Antes de aplicar cualquier algoritmo, SIEMPRE exploramos los datos.
Es como un médico que revisa los signos vitales antes de recetar.

### 📦 Celda 3: Dimensiones y estructura del dataset

In [ ]:
# ============================================
# CELDA 3: Exploración inicial del dataset
# ============================================

# ─── ¿Cuántos registros y columnas tenemos? ───
print("📊 Dimensiones (filas, columnas):", df_scada.shape)
#   ↑ '.shape' es un ATRIBUTO (no función, no lleva paréntesis).
#   ↑ Devuelve una tupla: (número_de_filas, número_de_columnas).
#   ↑ Esperamos (1000, 3): 1,000 registros y 3 columnas.

# ─── ¿Qué tipo de dato tiene cada columna? ───
print("\n📋 Tipos de datos:")
print(df_scada.dtypes)
#   ↑ '.dtypes' = atributo que muestra el tipo de cada columna.
#   ↑ 'int64' = número entero (como Registro_ID).
#   ↑ 'float64' = número decimal (como WHP_psi y Qo_bpd).
#   ↑ 'object' = texto. Si WHP_psi fuera 'object' en vez de 'float64',
#   ↑   significaría que hay letras mezcladas con números → ¡problema!

# ─── Estadísticas rápidas ───
print("\n📈 Estadísticas descriptivas:")
df_scada.describe()
#   ↑ '.describe()' = genera un resumen estadístico automático:
#   ↑   count = cuántos valores hay
#   ↑   mean = promedio
#   ↑   std = desviación estándar (qué tan dispersos están los datos)
#   ↑   min = valor mínimo
#   ↑   25%, 50%, 75% = percentiles (el 50% es la mediana)
#   ↑   max = valor máximo
#   ↑ Útil para detectar valores extremos de un vistazo.


### 🔍 ¿Qué buscar aquí?

- **`shape`**: Esperamos (1000, 3). Si ves algo muy distinto, el CSV no se cargó bien.
- **`dtypes`**: WHP_psi y Qo_bpd DEBEN ser `float64`. Si son `object`, hay texto infiltrado.
- **`describe()`**: Mira `max` de Qo_bpd. Si es 99,999 — ya encontraste un spike.
  Mira `min` de WHP_psi. Si es negativo — hay un código de error.

---
## 📊 PARTE 4: Visualizando los datos

Una imagen vale más que mil números. Vamos a graficar **todos los registros**
en un plano 2D: presión en cabeza (WHP) en el eje X, caudal de crudo (Qo) en
el eje Y. Cada punto es un registro SCADA de un minuto.

> 🎯 **Objetivo visual:** encontrar puntos que estén **lejos** de la nube
> principal. Esos puntos "volando" son candidatos a ser ruido.

### 📦 Celda 4: Scatterplot — Presión vs Caudal

In [ ]:
# ============================================
# CELDA 4: Visualizar la distribución conjunta
# ============================================

# ─── Gráfico de dispersión ───
sns.scatterplot(
    data=df_scada,          # ← DataFrame fuente (nuestra tabla)
    x='WHP_psi',            # ← Columna para el eje X: Presión en Cabeza
    y='Qo_bpd',             # ← Columna para el eje Y: Caudal de Crudo
    color='red',            # ← TODOS los puntos en rojo (aún no distinguimos clusters)
    alpha=0.6               # ← Transparencia: 0=invisible, 1=opaco.
                            #   0.6 permite ver zonas donde se solapan muchos puntos
)

# ─── Etiquetas y título ───
plt.title('Distribución Conjunta: Presión en Cabeza vs Caudal de Crudo')
#   ↑ Título arriba del gráfico. Describe QUÉ estamos viendo.

plt.xlabel('Presión en Cabeza - WHP (psi)')
#   ↑ Etiqueta del eje X.

plt.ylabel('Caudal de Crudo - Qo (bpd)')
#   ↑ Etiqueta del eje Y.

plt.show()
#   ↑ ¡RENDERIZA el gráfico! Sin esto, en Colab el gráfico NO aparece.
#   ↑ En Jupyter clásico sí aparece sin plt.show(), pero en Colab es necesario.


### 🔍 ¿Qué deberías observar?

Una **nube densa** de puntos concentrados en una región del gráfico (zona de
operación normal del pozo). Esa es la "zona de alta densidad".

Algunos puntos aparecerán **aislados**, lejos de la nube principal. Esos son
los candidatos a **ruido** — registros donde la combinación presión-caudal es
sospechosa.

> 🧠 **Pregunta para reflexionar:** ¿Podrías identificar manualmente cuáles
> puntos son anómalos? Probablemente sí, con 1,000 puntos y paciencia. Pero
> ¿y si fueran 1 millón? Por eso necesitamos DBSCAN.

---
## 📏 PARTE 5: Escalando los datos (¡IGUAL que en K-Means!)

**Regla de oro repetida:** Sin escalar, DBSCAN NO funciona. ¿Por qué?

El caudal (Qo) está en **miles** (1000-2000 bpd). La presión (WHP) está en
**cientos** (200-400 psi). Si no escalamos, el caudal **domina** el cálculo
de distancias y la presión se vuelve invisible para el algoritmo.

Usamos `StandardScaler` para transformar cada variable a:
- Media = 0
- Desviación estándar = 1

Después del escalado, **+1 en caudal pesa lo mismo que +1 en presión**.

### 📦 Celda 5: Escalar con StandardScaler

In [ ]:
# ============================================
# CELDA 5: Escalar las variables (Z-Score)
# ============================================

# ─── Paso 1: Definir qué columnas usaremos ───
features = ['Qo_bpd', 'WHP_psi']
#   ↑ Lista con los NOMBRES de las columnas que DBSCAN usará.
#   ↑ Solo 2 columnas en este dataset, pero podrían ser 5, 10 o más.
#   ↑ CADA columna en esta lista = UNA dimensión que DBSCAN "ve".

# ─── Paso 2: Crear el objeto escalador ───
scaler = StandardScaler()
#   ↑ 'StandardScaler()' crea el escalador Z-Score.
#   ↑ En este momento NO ha hecho nada aún — solo creamos la "máquina".
#   ↑ Es como encender el equipo pero todavía no medir nada.

# ─── Paso 3: Calcular media, desviación Y transformar ───
X_scaled = scaler.fit_transform(df_scada[features])
#   ↑ 'df_scada[features]' = selecciona SOLO las columnas en la lista 'features'.
#   ↑   Esto devuelve un DataFrame con 2 columnas: Qo_bpd y WHP_psi.
#   ↑ '.fit_transform()' hace DOS cosas en un solo paso:
#   ↑   1. .fit()    = calcula la media (μ) y desviación estándar (σ) de cada columna
#   ↑   2. .transform() = aplica la fórmula z = (x - μ) / σ a cada valor
#   ↑ 'X_scaled' = resultado: un array de NumPy con media=0 y desviación=1.
#   ↑   Ya NO es un DataFrame de pandas — es un array numérico puro.

# ─── Paso 4: Verificar que el escalado funcionó ───
print("✅ Datos escalados. Forma:", X_scaled.shape)
print("   Media de Qo (debe ser ≈0):", X_scaled[:, 0].mean().round(6))
#   ↑ '[:, 0]' = TODAS las filas (:), columna 0 (Qo_bpd, primera en la lista).
#   ↑ '.mean()' = calcula el promedio. Debe ser aproximadamente 0.
print("   Desviación de Qo (debe ser ≈1):", X_scaled[:, 0].std().round(6))
#   ↑ '.std()' = calcula la desviación estándar. Debe ser aproximadamente 1.
print("   Media de WHP (debe ser ≈0):", X_scaled[:, 1].mean().round(6))
print("   Desviación de WHP (debe ser ≈1):", X_scaled[:, 1].std().round(6))


### 🔍 ¿Qué significa "escalar"?

| Antes (datos crudos) | Después (Z-Score) |
|---|---|
| Qo: 1200, 1850, 950... | Qo: -0.8, +1.2, -1.5... |
| WHP: 350, 280, 410... | WHP: +0.4, -1.1, +1.8... |

Los números originales (1200, 350) se convierten en "¿qué tan lejos de la
media está este valor?". +1.2 significa "1.2 desviaciones estándar por
encima del promedio".

> ✅ **Check:** Las medias deben ser ≈0 y las desviaciones ≈1. Si no lo son,
> algo falló en el escalado.

---
## 🎛️ PARTE 6: Calibrando Epsilon (ε) — ¡La parte MÁS importante!

DBSCAN necesita que le digas **qué tan lejos** buscar vecinos. Eso es **ε**
(epsilon). NO se adivina — se calcula con el **gráfico de k-distancias**.

### 🧠 La lógica:
1. Para cada punto, mido la distancia a su **4to vecino más cercano**
2. Ordeno todas esas distancias de menor a mayor
3. Grafico — busco **EL CODO** donde la curva se dispara
4. El valor Y en el codo = mi ε óptimo

> 🔑 **MinPts = 4** significa: "necesito el punto + 3 vecinos para ser core".
> Buscamos la distancia al 4to vecino porque más allá no necesito mirar.

### 📦 Celda 6: Calcular distancias a vecinos más cercanos

In [ ]:
# ============================================
# CELDA 6: Calibrar Epsilon — K-Distance Graph
# ============================================

# ─── Paso 1: Calcular los 4 vecinos más cercanos de cada punto ───
neighbors = NearestNeighbors(n_neighbors=4)
#   ↑ Crea el objeto que calculará vecinos.
#   ↑ 'n_neighbors=4' = para cada punto, encuentra sus 4 vecinos más cercanos.
#   ↑ ¿Por qué 4? Porque MinPts=4. El 4to vecino es el más lejano que me importa.

neighbors_fit = neighbors.fit(X_scaled)
#   ↑ '.fit()' = "aprende" la estructura del espacio.
#   ↑ Indexa todos los puntos para búsquedas rápidas.
#   ↑ NO modifica los datos — solo los organiza internamente.

distances, indices = neighbors_fit.kneighbors(X_scaled)
#   ↑ '.kneighbors()' = para CADA punto, calcula sus 4 vecinos más cercanos.
#   ↑ Devuelve DOS cosas:
#   ↑   'distances' = matriz de 1000 filas × 4 columnas con las distancias
#   ↑   'indices'   = matriz de 1000 filas × 4 columnas con los índices de los vecinos
#   ↑   (no usaremos 'indices' — lo recibimos pero lo ignoramos)

# ─── Paso 2: Extraer SOLO la distancia al 4to vecino ───
distances_sorted = np.sort(distances[:, 3])
#   ↑ 'distances[:, 3]' = TODAS las filas (:), columna 3 (índice 3 = 4ta columna).
#   ↑   Python cuenta desde 0: columna 0=1er vecino, 1=2do, 2=3ro, 3=4to.
#   ↑ 'np.sort()' = ordena esas distancias de MENOR a MAYOR.
#   ↑   Las pequeñas (zonas densas) quedan al inicio; las grandes (ruido) al final.

# ─── Paso 3: Graficar para encontrar el codo ───
plt.figure(figsize=(10, 5))
#   ↑ 'figsize=(10,5)' = ancho 10 pulgadas, alto 5 pulgadas.

plt.plot(distances_sorted, linewidth=2)
#   ↑ Grafica la curva. Eje X = puntos ordenados. Eje Y = distancia al 4to vecino.
#   ↑ 'linewidth=2' = línea más gruesa para mejor visibilidad.

plt.axhline(y=0.25, color='red', linestyle='--', linewidth=2, label='ε recomendado = 0.25')
#   ↑ 'axhline' = "axis horizontal line" — dibuja una línea horizontal.
#   ↑ 'y=0.25' = la línea cruza el eje Y en 0.25 (nuestro ε calibrado).
#   ↑ 'color=red', 'linestyle=--' = línea roja punteada.
#   ↑ 'label=...' = etiqueta para la leyenda.

plt.title('Gráfico de K-Distancia: Calibración de Epsilon (ε)')
plt.xlabel('Puntos ordenados por distancia')
plt.ylabel('Distancia al 4to vecino más cercano')
plt.legend()
#   ↑ Muestra la cajita con las etiquetas de las líneas.

plt.grid(True, alpha=0.3)
#   ↑ Agrega una cuadrícula suave para leer valores más fácilmente.

plt.show()


### 🔍 ¿Cómo leer este gráfico?

```
distancia
   ↑
   |                    .  ← ruido (distancia ENORME)
   |                 .
   |              .
   |           .  ← EL CODO (ε ≈ 0.25)
   |        .
   |     .
   |  .  ← zona densa (distancia PEQUEÑA)
   └──────────────────────→ puntos ordenados
```

- **Zona plana (izquierda):** Puntos en zonas densas. Sus vecinos están cerca.
- **El CODO:** Donde la pendiente cambia bruscamente. ¡Ahí pones ε!
- **Zona empinada (derecha):** Puntos aislados. Sus vecinos están lejísimos.

> ⚠️ **¿Qué pasa si elijo mal?**
> - ε muy pequeño (ej. 0.05) → casi todo es ruido.
> - ε muy grande (ej. 2.0) → todo se fusiona en un solo cluster.

---
## 🤖 PARTE 7: Entrenando DBSCAN

¡Llegó el momento! Con ε=0.25 y MinPts=4, entrenamos DBSCAN.
En UNA sola línea de código, el algoritmo:
1. Encuentra zonas densas → las agrupa en clusters (0, 1, 2...)
2. Detecta puntos aislados → los marca como **RUIDO (-1)**

### 📦 Celda 7: Entrenar DBSCAN

In [ ]:
# ============================================
# CELDA 7: Entrenar DBSCAN
# ============================================

# ─── Paso 1: Crear y entrenar DBSCAN ───
dbscan = DBSCAN(eps=0.25, min_samples=4)
#   ↑ Crea el algoritmo DBSCAN con DOS parámetros:
#   ↑   'eps=0.25' = radio de búsqueda (obtenido del codo en el gráfico k-distance).
#   ↑     Está en unidades Z-Score: 0.25 desviaciones estándar.
#   ↑   'min_samples=4' = necesito al menos 4 puntos en radio ε para ser core.
#   ↑     Incluye el punto mismo + 3 vecinos.

df_scada['Cluster'] = dbscan.fit_predict(X_scaled)
#   ↑ 'fit_predict()' = entrena Y predice en UN solo paso.
#   ↑   1. .fit() analiza la densidad de cada punto.
#   ↑   2. .predict() asigna: 0, 1, 2... = cluster al que pertenece.
#   ↑                        -1 = RUIDO (no pertenece a ningún cluster).
#   ↑ 'df_scada['Cluster'] = ...' crea una NUEVA columna en el DataFrame
#   ↑   con la etiqueta asignada a cada registro.

# ─── Paso 2: Ver resultados ───
print("📊 Conteo de puntos por cluster:")
print(df_scada['Cluster'].value_counts())
#   ↑ '.value_counts()' = cuenta cuántos registros hay en cada cluster.
#   ↑ Ejemplo de salida:
#   ↑    0    920  ← 920 puntos en el cluster 0 (datos sanos)
#   ↑   -1    80  ←  80 puntos son RUIDO (anomalías)
#   ↑   Name: Cluster, dtype: int64

print(f"\n🔴 Puntos de RUIDO (-1): {(df_scada['Cluster'] == -1).sum()}")
#   ↑ '(df_scada['Cluster'] == -1)' = Serie de True/False donde Cluster es -1.
#   ↑ '.sum()' = cuenta los True (True=1, False=0).
print(f"🟢 Puntos en clusters: {(df_scada['Cluster'] != -1).sum()}")
#   ↑ '!= -1' = "distinto de -1" — todos los que SÍ pertenecen a algún cluster.


### 🔍 Interpretación

- **Cluster 0:** la mayoría de los puntos — datos SANOS, operación normal.
- **Cluster -1:** los puntos que DBSCAN no pudo agrupar — ANOMALÍAS.

> 🧠 **El -1 NO es un error.** Es INFORMACIÓN VALIOSA. Significa:
> "Este registro no se parece a nada. Investígalo."

> ⚠️ Si ves más del 30% de puntos como -1, ε es muy pequeño. Si ves 0 puntos
> como -1, ε es muy grande.

---
## 🎨 PARTE 8: Visualizando el resultado

Ahora graficamos los MISMOS datos que en la Celda 4, pero esta vez
**coloreados por cluster**. Los puntos de ruido (-1) aparecerán en
un color distinto — típicamente gris o negro.

### 📦 Celda 8: Scatterplot coloreado por cluster

In [ ]:
# ============================================
# CELDA 8: Visualizar resultados de DBSCAN
# ============================================

# ─── Gráfico coloreado por cluster ───
plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=df_scada,          # ← Mismo DataFrame de siempre
    x='WHP_psi',            # ← Eje X: Presión en Cabeza
    y='Qo_bpd',             # ← Eje Y: Caudal de Crudo
    hue='Cluster',          # ← ¡ESTA ES LA MAGIA! Colorea según la columna 'Cluster'
                            #   0 = un color, -1 = otro color distinto
    palette='tab10',        # ← Paleta de 10 colores bien diferenciados
    alpha=0.8               # ← 80% de opacidad
)

plt.title('Detección Multivariada de Anomalías SCADA con DBSCAN')
plt.xlabel('Presión en Cabeza - WHP (psi)')
plt.ylabel('Caudal de Crudo - Qo (bpd)')
plt.legend(title='Cluster')
#   ↑ Muestra la leyenda con el título "Cluster".
#   ↑ Cluster 0 = color azul, Cluster -1 = gris/negro.

plt.show()


### 🔍 ¿Qué deberías observar?

- **Cluster 0 (azul):** la nube DENSA de puntos — **DATOS SANOS**.
  Operación normal del pozo.
- **Cluster -1 (gris/negro):** puntos DISPERSOS, lejos de la nube — **ANOMALÍAS**.
  Estos son los registros donde la combinación presión-caudal es
  físicamente incoherente.

> 🛢️ **Interpretación petrolera:** Cada punto -1 es un candidato a:
> - Falla de telemetría (sensor desconectado)
> - Spike eléctrico (interferencia en el cableado)
> - Dato congelado (transmisor trabado)
> - Lectura físicamente imposible

---
## 🔎 PARTE 9: Investigando las anomalías detectadas

DBSCAN nos dijo QUÉ registros son sospechosos. Ahora vamos a VERLOS
de cerca para decidir qué hacer con ellos.

### 📦 Celda 9: Examinar los puntos de ruido

In [ ]:
# ============================================
# CELDA 9: Examinar los puntos de RUIDO (-1)
# ============================================

# ─── Filtrar SOLO los registros anómalos ───
anomalias = df_scada[df_scada['Cluster'] == -1]
#   ↑ 'df_scada['Cluster'] == -1' = filtro: True donde Cluster es -1.
#   ↑ 'df_scada[ ... ]' = aplica el filtro: solo filas donde es True.
#   ↑ 'anomalias' = nuevo DataFrame con SOLO los registros anómalos.

print(f"🔴 Total de anomalías detectadas: {len(anomalias)} de {len(df_scada)} registros")
print(f"   ({len(anomalias)/len(df_scada)*100:.1f}% del total)\n")
#   ↑ 'len()' = número de filas.
#   ↑ ':.1f' = formatea el número con 1 decimal.

print("📋 Primeras 10 anomalías:")
anomalias.head(10)
#   ↑ Muestra las primeras 10 filas del DataFrame de anomalías.


### 🔍 Preguntas para reflexionar

1. ¿Las anomalías tienen valores EXTREMOS en alguna variable?
2. ¿O tienen valores "normales" pero en una COMBINACIÓN imposible?
3. ¿Qué harías con estos registros en tu trabajo diario?
   - ¿Los eliminarías?
   - ¿Los interpolarías?
   - ¿Investigarías el sensor?

> 💡 **DBSCAN no decide por ti.** Te dice "esto es raro". La decisión de
> ingeniería — eliminar, interpolar, investigar — es TUYA.

---
## 📋 RECAP: El Pipeline DBSCAN Completo

```
┌─────────────────────────────────────────────────────┐
│  1. IMPORTAR herramientas   →  import sklearn...    │
│  2. CARGAR datos            →  pd.read_csv()        │
│  3. EXPLORAR                →  .shape, .describe()  │
│  4. VISUALIZAR (crudo)      →  sns.scatterplot()    │
│  5. ESCALAR                 →  StandardScaler()     │
│  6. CALIBRAR ε              →  NearestNeighbors()   │
│  7. ENTRENAR DBSCAN         →  fit_predict()        │
│  8. VISUALIZAR (resultado)  →  scatterplot(hue=)    │
│  9. INVESTIGAR anomalías    →  df[df['Cluster']==-1]│
└─────────────────────────────────────────────────────┘
```

### ✅ Lo que aprendiste hoy

- DBSCAN agrupa por **densidad**, no por distancia a centroides
- Los puntos que no pertenecen a ninguna zona densa son **RUIDO (-1)**
- **ε (epsilon)** se calibra con el gráfico de k-distancias, NO se adivina
- El **escalado** es igual de obligatorio que en K-Means
- DBSCAN funciona con **2, 5, 10 o más variables**
- Las anomalías detectadas requieren **decisión de ingeniería**

> 🚀 **Siguiente paso:** Mañana, Día 3 — PCA para reducir dimensiones
> y visualizar clusters en datos de alta dimensionalidad.
